# Limpieza y preparación de datos

En este tutorial trabajaremos con una base pequeña de clientes y compras. El objetivo es practicar una secuencia básica de limpieza: detectar valores faltantes, revisar duplicados, corregir inconsistencias simples y examinar posibles valores atípicos.

## Requisitos

Para desarrollar este tutorial necesitarás:

* Importar archivos con `pandas`.
* Consultar filas y columnas de un `DataFrame`.
* Utilizar operaciones básicas de filtrado.

## Objetivos

Al final de este tutorial podrás:

**1.** Identificar valores faltantes y cuantificarlos. <br>
**2.** Detectar y eliminar duplicados simples. <br>
**3.** Revisar valores atípicos con reglas básicas. <br>
**4.** Construir una versión más limpia del dataset.

## 1. Cargar los datos

Empezaremos importando `pandas` y leyendo el archivo de trabajo.

In [1]:
from pathlib import Path
import pandas as pd

ruta = Path("Archivos") / "clientes_ventas_sucio.csv"
df = pd.read_csv(ruta)
df

,id_cliente,nombre,ciudad,edad,ingreso_mensual,fecha_registro,compras_ultimo_mes,puntaje_satisfaccion
0,1,Ana Ruiz,Bogota,29.0,2500000.0,2025-01-15,3.0,4.5
1,2,Carlos Perez,Medellin,35.0,3200000.0,2025-01-18,5.0,4.8
2,3,Luisa Gomez,Cali,NaN,2800000.0,2025-01-20,2.0,4.1
3,4,Juan Torres,Bogotá,42.0,NaN,2025-02-01,4.0,4.0
4,5,Maria Lopez,bogota,150.0,3000000.0,2025-02-03,3.0,3.9
5,6,Pedro Silva,Barranquilla,31.0,9800000.0,2025-02-10,18.0,4.7
6,7,Sofia Diaz,Medellin,27.0,2600000.0,2025-02-12,NaN,4.6
7,8,Andres Castro,Cali,38.0,2900000.0,2025-02-15,4.0,NaN
8,9,Camila Naranjo,Bucaramanga,33.0,3100000.0,2025-02-20,5.0,4.9
9,10,Diego Marin,Cartagena,29.0,2750000.0,2025-02-21,3.0,4.2


El `DataFrame` tiene algunos problemas intencionales para practicar: celdas vacías, una fila duplicada, ciudades escritas de manera inconsistente y algunos valores que merecen revisión.

## 2. Inspección inicial

Antes de limpiar conviene entender el tamaño y la estructura del dataset.

In [2]:
print("Dimensiones:", df.shape)
print("\nTipos de datos:")
print(df.dtypes)

Dimensiones: (14, 8)

Tipos de datos:
id_cliente                int64
nombre                   object
ciudad                   object
edad                    float64
ingreso_mensual         float64
fecha_registro           object
compras_ultimo_mes      float64
puntaje_satisfaccion    float64
dtype: object


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id_cliente            14 non-null     int64  
 1   nombre                14 non-null     object 
 2   ciudad                13 non-null     object 
 3   edad                  13 non-null     float64
 4   ingreso_mensual       13 non-null     float64
 5   fecha_registro        14 non-null     object 
 6   compras_ultimo_mes    13 non-null     float64
 7   puntaje_satisfaccion  13 non-null     float64
dtypes: float64(4), int64(1), object(3)
memory usage: 1.0+ KB


## 3. Revisar valores faltantes

El método `isna()` identifica celdas vacías. Si sumamos por columna, obtenemos cuántos faltantes hay en cada variable.

In [4]:
faltantes = df.isna().sum().sort_values(ascending=False)
faltantes

ciudad                  1
edad                    1
ingreso_mensual         1
compras_ultimo_mes      1
puntaje_satisfaccion    1
id_cliente              0
nombre                  0
fecha_registro          0
dtype: int64

Observa que algunas columnas tienen pocos faltantes. En un ejemplo pedagógico como este vamos a usar reglas sencillas:

* Para texto, llenaremos con una categoría explícita.
* Para números, usaremos la mediana cuando tenga sentido.

In [5]:
df_limpio = df.copy()

df_limpio["ciudad"] = df_limpio["ciudad"].fillna("Desconocida")
df_limpio["edad"] = df_limpio["edad"].fillna(df_limpio["edad"].median())
df_limpio["ingreso_mensual"] = df_limpio["ingreso_mensual"].fillna(df_limpio["ingreso_mensual"].median())
df_limpio["compras_ultimo_mes"] = df_limpio["compras_ultimo_mes"].fillna(df_limpio["compras_ultimo_mes"].median())
df_limpio["puntaje_satisfaccion"] = df_limpio["puntaje_satisfaccion"].fillna(df_limpio["puntaje_satisfaccion"].median())

df_limpio.isna().sum()

id_cliente              0
nombre                  0
ciudad                  0
edad                    0
ingreso_mensual         0
fecha_registro          0
compras_ultimo_mes      0
puntaje_satisfaccion    0
dtype: int64

## 4. Detectar duplicados

`duplicated()` marca las filas repetidas. Si una observación aparece más de una vez, debemos revisar si realmente es redundante.

In [6]:
df_limpio[df_limpio.duplicated()]

,id_cliente,nombre,ciudad,edad,ingreso_mensual,fecha_registro,compras_ultimo_mes,puntaje_satisfaccion
10,10,Diego Marin,Cartagena,29.0,2750000.0,2025-02-21,3.0,4.2


In [7]:
df_limpio = df_limpio.drop_duplicates()
print("Filas después de eliminar duplicados:", len(df_limpio))

Filas después de eliminar duplicados: 13


## 5. Estandarizar texto

Una base puede tener la misma ciudad escrita de varias maneras: con mayúsculas, tildes o espacios extra. Esto dificulta conteos y agrupaciones.

In [8]:
print("Ciudades originales:")
print(sorted(df_limpio["ciudad"].unique()))

Ciudades originales:
['Barranquilla', 'Bogota', 'Bogotá', 'Bucaramanga', 'Cali', 'Cartagena', 'Desconocida', 'Medellin', 'Medellin ', 'Medellín', 'bogota']


In [9]:
df_limpio["ciudad"] = (
    df_limpio["ciudad"]
    .str.strip()
    .str.lower()
    .replace({
        "bogotá": "bogota",
        "medellín": "medellin"
    })
    .str.title()
)

print("Ciudades estandarizadas:")
print(sorted(df_limpio["ciudad"].unique()))

Ciudades estandarizadas:
['Barranquilla', 'Bogota', 'Bucaramanga', 'Cali', 'Cartagena', 'Desconocida', 'Medellin']


## 6. Revisar valores atípicos

Para `edad` e `ingreso_mensual` haremos dos revisiones complementarias:

* una regla de negocio simple para edades imposibles o poco plausibles;
* una regla estadística con rango intercuartílico para ingresos.

In [10]:
df_limpio[df_limpio["edad"] > 100]

,id_cliente,nombre,ciudad,edad,ingreso_mensual,fecha_registro,compras_ultimo_mes,puntaje_satisfaccion
4,5,Maria Lopez,Bogota,150.0,3000000.0,2025-02-03,3.0,3.9


In [11]:
mediana_edad = df_limpio.loc[df_limpio["edad"].between(18, 100), "edad"].median()
df_limpio.loc[~df_limpio["edad"].between(18, 100), "edad"] = mediana_edad
df_limpio["edad"].describe()

count    13.000000
mean     33.461538
std       4.806513
min      27.000000
25%      29.000000
50%      33.000000
75%      36.000000
max      42.000000
Name: edad, dtype: float64

In [12]:
q1 = df_limpio["ingreso_mensual"].quantile(0.25)
q3 = df_limpio["ingreso_mensual"].quantile(0.75)
iqr = q3 - q1
limite_superior = q3 + 1.5 * iqr

ingresos_atipicos = df_limpio[df_limpio["ingreso_mensual"] > limite_superior]
ingresos_atipicos

,id_cliente,nombre,ciudad,edad,ingreso_mensual,fecha_registro,compras_ultimo_mes,puntaje_satisfaccion
5,6,Pedro Silva,Barranquilla,31.0,9800000.0,2025-02-10,18.0,4.7


En este ejemplo aplicaremos una solución conservadora: recortar valores extremos al límite superior calculado. En proyectos reales esta decisión debe justificarse con conocimiento del negocio.

In [13]:
df_limpio["ingreso_mensual"] = df_limpio["ingreso_mensual"].clip(upper=limite_superior)
df_limpio["ingreso_mensual"].describe()

count    1.300000e+01
mean     2.773077e+06
std      8.971286e+05
min      0.000000e+00
25%      2.750000e+06
50%      2.900000e+06
75%      3.150000e+06
max      3.750000e+06
Name: ingreso_mensual, dtype: float64

## 7. Verificación rápida de calidad

Después de limpiar, conviene revisar si se resolvieron los problemas principales.

In [14]:
resumen_calidad = pd.DataFrame({
    "faltantes": df_limpio.isna().sum(),
    "tipo_dato": df_limpio.dtypes.astype(str)
})

resumen_calidad

,faltantes,tipo_dato
id_cliente,0,int64
nombre,0,object
ciudad,0,object
edad,0,float64
ingreso_mensual,0,float64
fecha_registro,0,object
compras_ultimo_mes,0,float64
puntaje_satisfaccion,0,float64


In [15]:
print("Número de duplicados restantes:", df_limpio.duplicated().sum())
print("Edades fuera de rango:", (~df_limpio["edad"].between(18, 100)).sum())

Número de duplicados restantes: 0
Edades fuera de rango: 0


## 8. Guardar la versión limpia

Es buena práctica conservar el archivo original y exportar la versión preparada con otro nombre.

In [16]:
salida = Path("Archivos") / "clientes_ventas_limpio.csv"
df_limpio.to_csv(salida, index=False)
print("Archivo guardado en:", salida)

Archivo guardado en: Archivos\clientes_ventas_limpio.csv


## Idea final

La limpieza no es un conjunto de trucos sueltos. Es una secuencia de decisiones justificadas para dejar los datos en mejores condiciones de análisis. Mientras más claro sea el criterio, más fácil será explicar y reproducir el proceso.